# RAG Pipeline Validation Notebook

This notebook enables testing and validation of the Rafcio Assistant RAG pipeline, including:
1. **Configuration & Connectivity**: Checking if LM Studio and ChromaDB are responsive.
2. **Vector Store**: Verifying document ingestion and retrieval.
3. **Hybrid Search**: Comparing Vector vs. BM25 vs. Combined results.
4. **RAG Chain**: Testing end-to-end question answering with the LLM.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys

# Ensure project root is in path
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

from src.config import settings
from src.embeddings import LMStudioEmbeddings
from src.rag_chain import RAGChain
from src.retrieval import HybridRetriever
from src.vectorstore import VectorStoreManager

print(f"API URL: {settings.api_url}")
print(f"LLM Model: {settings.llm_model}")
print(f"Embedding Model: {settings.embedding_model}")

API URL: http://127.0.0.1:1234
LLM Model: openai/gpt-oss-20b
Embedding Model: text-embedding-qwen3-embedding-8b


## 1. Initialize Components

We'll initialize the embeddings, the vector store adapter, and the hybrid retriever. 

In [2]:
# Initialize Embeddings
embeddings = LMStudioEmbeddings()

# Initialize Vector Store Manager
manager = VectorStoreManager(
    provider=settings.vector_store_provider,
    embedding_model=embeddings,
    persist_directory=os.path.join(project_root, "data/chroma_db"),
    collection_name=settings.vector_store_collection,
)
adapter = manager.get_adapter()

# Initialize Retriever
retriever = HybridRetriever(vector_adapter=adapter, top_k=settings.top_k_results)

# Initialize RAG Chain
rag_chain = RAGChain(retriever=retriever)

print("Components initialized successfully!")

Components initialized successfully!


## 2. Test Retrieval

Let's test if we can find relevant documents in the vector store.

In [3]:
query = "What is the main architecture of this assistant?"
results = retriever.search(query)

print(f"Query: {query}\n")
for i, res in enumerate(results):
    print(f"--- Result {i + 1} ---")
    print(f"Source: {res['metadata'].get('source', 'Unknown')}")
    print(f"Text: {res['text'][:300]}...")
    print()

Query: What is the main architecture of this assistant?

--- Result 1 ---
Source: knowledge_base/Introduction to Agents from Google.pdf
Text: Introduction to Agents and Agent architectures
November 2025
19
Core Agent Architecture: Model, Tools, 
and Orchestration
We know what an agent does and how it can scale. But how do we actually build it? 
The transition from concept to code lies in the specific architectural design of its three 
cor...

--- Result 2 ---
Source: knowledge_base/Introduction to Agents from Google.pdf
Text: is a relentless loop of assembling context, prompting the model, observing the result, 
and then re-assembling a context for the next step. The context may include system 
instructions, user input, session history, long term memories, grounding knowledge from 
authoritative sources, what tools could...

--- Result 3 ---
Source: knowledge_base/Introduction to Agents from Google.pdf
Text: 15. https://ai.google.dev/gemini-api/docs/function-calling
16. https://github.

## 3. Test RAG Chain

Now let's ask a question and see how the LLM synthesizes the answer using the retrieved context.

In [4]:
question = "Explain the role of HybridRetriever in this project."
answer = rag_chain.invoke(question)

print(f"Question: {question}\n")
print(f"Answer: {answer}")

Question: Explain the role of HybridRetriever in this project.

Answer: **HybridRetriever – the heart of the retrieval step**

In this project the *HybridRetriever* is the component that actually pulls the most useful information from the knowledge base before it is fed to the language model. It does this by combining two complementary search techniques:

| What it does | How it works |
|--------------|--------------|
| **Keyword (BM25) search** | Uses a traditional inverted‑index to find passages that contain the exact query terms. This is fast and good for highly specific queries. |
| **Semantic vector search** | Uses an embedding model (e.g., `text-embedding-bge-m3`) to embed both the query and all passages in a vector database (ChromaDB). The nearest‑neighbour search retrieves passages that are semantically similar even if they don’t share exact words. |
| **Fusion of results** | The two ranked lists are merged with *Reciprocal Rank Fusion (RRF)*, a lightweight algorithm that gives

## 4. Interactive Sandbox

Use the cell below to ask arbitrary questions to the system.

In [5]:
while True:
    user_input = input("Ask a question (or type 'exit' to stop): ")
    if user_input.lower() in ["exit", "quit"]:
        break

    try:
        print(f"\nProcessing: {user_input}...")
        ans = rag_chain.invoke(user_input)
        print(f"\nResponse:\n{ans}\n")
        print("-" * 50)
    except Exception as e:
        print(f"Error: {e}")


Processing: What google suggest as agentic tools ...

Response:
Google recommends a handful of built‑in “agentic” tools that let an LLM reach out to the web and other services.  In particular:

| Tool | What it does |
|------|--------------|
| **google_search** (the Google Search API) | Executes a web search and returns the top results. |
| **google_places** | Looks up local places (e.g., coffee shops, restaurants) and can filter by rating or other attributes. |
| **Gemini function‑calling** | Allows the model to call arbitrary functions (including the above tools) via the Gemini API’s function‑calling interface. |

These are the primary agentic tools that Google highlights for building goal‑oriented agents.

--------------------------------------------------

Processing: what agents frameworks to use ...

Response:
The context you provided only mentions **Vertex AI Agent Builder** as a concrete agent framework that offers an easy “deploy” command and a dedicated platform for deployin